# 📊 PHÂN TÍCH DỮ LIỆU HỌC TẬP - LỚP 58KTPM (K58KTP)
## Jupyter Notebook - Python Data Science (Pandas + Matplotlib + Scikit-learn)

> **Dữ liệu:** Bảng tổng hợp điểm từ kỳ 1 năm nhất đến hết kỳ 2 năm 4  
> **Lớp:** 58KTPM – Khoa Công nghệ phần mềm  
> **Mục tiêu:** Phân tích điểm số, phân cụm sinh viên, xây dựng mô hình dự đoán học lực


## 1. 📦 Import thư viện

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

# Cấu hình hiển thị
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:.2f}'.format)

print("✅ Import thư viện thành công!")


## 2. 📂 Đọc & Làm sạch dữ liệu

In [ ]:
# Đọc file Excel gốc
df_raw = pd.read_excel('TỔNG_HỢP_ĐIỂM_K58KTP.xlsx', header=None)

# ── Trích xuất thông tin sinh viên ──
mssv  = df_raw.iloc[1, 3:].values
names = df_raw.iloc[2, 3:].values
ma_mon    = df_raw.iloc[4:, 1].values
ten_mon   = df_raw.iloc[4:, 2].values
scores_raw = df_raw.iloc[4:, 3:].values

# Hàm làm sạch điểm (thang 4, loại ngày tháng/chuỗi lạ)
def safe_score(x):
    try:
        v = float(x)
        return v if 0 <= v <= 4 else np.nan
    except:
        return np.nan

scores = np.vectorize(safe_score)(scores_raw).astype(float)

# ── Tạo DataFrame sinh viên ──
df_sv = pd.DataFrame(scores.T, columns=ma_mon)
df_sv.insert(0, 'MSSV', mssv)
df_sv.insert(1, 'Ho_Ten', names)

# Lọc bỏ cột không hợp lệ
valid_mask = [str(m) not in ('nan','None') for m in ma_mon]
df_sv = df_sv[['MSSV','Ho_Ten'] + list(ma_mon[valid_mask])]

# Bỏ sinh viên không có MSSV hợp lệ
df_sv = df_sv[df_sv['MSSV'].apply(lambda x: str(x).startswith('K2'))]
df_sv = df_sv.reset_index(drop=True)

print(f"✅ Đã đọc dữ liệu: {len(df_sv)} sinh viên, {len(ma_mon)} môn học")
df_sv[['MSSV','Ho_Ten']].head(10)


In [ ]:
# ── Tạo từ điển tên môn ──
mon_dict = dict(zip(ma_mon[valid_mask], ten_mon[valid_mask]))

# ── Tính GPA trung bình mỗi sinh viên ──
mon_cols = list(ma_mon[valid_mask])
df_sv['GPA'] = df_sv[mon_cols].mean(axis=1, skipna=True).round(2)
df_sv['So_mon_co_diem'] = df_sv[mon_cols].notna().sum(axis=1)

# ── Xếp loại học lực ──
def xep_loai(gpa):
    if gpa >= 3.6: return 'Xuất sắc'
    elif gpa >= 3.2: return 'Giỏi'
    elif gpa >= 2.5: return 'Khá'
    elif gpa >= 2.0: return 'Trung bình'
    else: return 'Yếu/Kém'

df_sv['Xep_loai'] = df_sv['GPA'].apply(xep_loai)

print("\n📊 Thống kê GPA toàn lớp:")
print(df_sv['GPA'].describe().round(2))
print("\n🏷️ Phân bố xếp loại:")
print(df_sv['Xep_loai'].value_counts())


## 3. 📈 Thống kê mô tả

In [ ]:
# ── 3.1 Phân bố GPA ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram GPA
axes[0].hist(df_sv['GPA'].dropna(), bins=15, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].axvline(df_sv['GPA'].mean(), color='red', linestyle='--', linewidth=2, label=f"TB: {df_sv['GPA'].mean():.2f}")
axes[0].set_title('Phân bố GPA toàn lớp', fontsize=13, fontweight='bold')
axes[0].set_xlabel('GPA (thang 4)')
axes[0].set_ylabel('Số sinh viên')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Pie xếp loại
colors_pie = ['#2ecc71','#27ae60','#f39c12','#e74c3c','#c0392b']
xep_loai_count = df_sv['Xep_loai'].value_counts()
wedges, texts, autotexts = axes[1].pie(
    xep_loai_count.values,
    labels=xep_loai_count.index,
    autopct='%1.1f%%',
    colors=colors_pie[:len(xep_loai_count)],
    startangle=90,
    wedgeprops=dict(edgecolor='white', linewidth=1.5)
)
for t in autotexts: t.set_fontsize(10)
axes[1].set_title('Tỷ lệ xếp loại học lực', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('bieu_do_gpa.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Đã lưu biểu đồ GPA")


In [ ]:
# ── 3.2 Điểm trung bình từng môn ──
mon_means = df_sv[mon_cols].mean().sort_values(ascending=False)
mon_names_short = [str(mon_dict.get(m, m))[:30] for m in mon_means.index]

fig, ax = plt.subplots(figsize=(14, 10))
colors_bar = ['#e74c3c' if v < 2.0 else '#f39c12' if v < 2.5 else '#2ecc71' for v in mon_means.values]
bars = ax.barh(range(len(mon_means)), mon_means.values, color=colors_bar, edgecolor='white', alpha=0.9)
ax.set_yticks(range(len(mon_means)))
ax.set_yticklabels(mon_names_short, fontsize=8.5)
ax.set_xlabel('Điểm trung bình (thang 4)', fontsize=11)
ax.set_title('Điểm trung bình từng môn học - Lớp 58KTPM', fontsize=13, fontweight='bold')
ax.axvline(2.0, color='red', linestyle='--', alpha=0.5, label='Ngưỡng 2.0')
ax.axvline(df_sv[mon_cols].mean().mean(), color='blue', linestyle=':', alpha=0.7,
           label=f"TB chung: {df_sv[mon_cols].mean().mean():.2f}")
ax.legend()
ax.grid(axis='x', alpha=0.3)
for i, (bar, val) in enumerate(zip(bars, mon_means.values)):
    ax.text(val + 0.03, i, f'{val:.2f}', va='center', fontsize=7.5)
plt.tight_layout()
plt.savefig('bieu_do_mon_hoc.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── 3.3 Top 10 SV có GPA cao nhất ──
top10 = df_sv.nlargest(10, 'GPA')[['MSSV','Ho_Ten','GPA','Xep_loai']].reset_index(drop=True)
top10.index = top10.index + 1
print("🏆 TOP 10 Sinh viên có GPA cao nhất:")
top10


In [ ]:
# ── 3.4 Môn học có nhiều SV điểm thấp nhất ──
mon_fail = (df_sv[mon_cols] < 2.0).sum().sort_values(ascending=False)
mon_fail_pct = (mon_fail / df_sv[mon_cols].notna().sum() * 100).round(1)

df_fail = pd.DataFrame({
    'Mã môn': mon_fail.head(10).index,
    'Tên môn': [str(mon_dict.get(m,''))[:35] for m in mon_fail.head(10).index],
    'Số SV < 2.0': mon_fail.head(10).values,
    'Tỷ lệ (%)': mon_fail_pct.head(10).values
})
print("⚠️ Top 10 môn học có nhiều sinh viên điểm dưới 2.0:")
df_fail


## 4. 🔥 Heatmap tương quan giữa các môn

In [ ]:
# Chọn các môn chuyên ngành CNPM (có đủ dữ liệu)
mon_cnpm = [m for m in mon_cols if m.startswith('TEE') or m.startswith('WSH')]
df_corr = df_sv[mon_cnpm].dropna(thresh=len(mon_cnpm)//2)
corr = df_corr.corr()

short_labels = [str(mon_dict.get(m,m))[:20] for m in corr.columns]
fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, vmin=-1, vmax=1,
            xticklabels=short_labels, yticklabels=short_labels,
            linewidths=0.4, ax=ax, annot_kws={'size': 7})
ax.set_title('Ma trận tương quan điểm số - Môn chuyên ngành', fontsize=13, fontweight='bold')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.savefig('heatmap_tuong_quan.png', dpi=150, bbox_inches='tight')
plt.show()


## 5. 🤖 Phân cụm sinh viên với K-Means (Yêu cầu bài tập)

> **Bài tập yêu cầu phân cụm lớp thành 3 nhóm** – dùng K-Means với PCA để trực quan hóa.


In [ ]:
# ── Chuẩn bị dữ liệu cho phân cụm ──
# Dùng các môn có đủ dữ liệu (>= 50% SV có điểm)
mon_cluster = [m for m in mon_cols if df_sv[m].notna().sum() >= len(df_sv)*0.5]
X_raw = df_sv[mon_cluster].copy()

# Điền NaN bằng điểm trung bình của từng môn
X_filled = X_raw.apply(lambda col: col.fillna(col.mean()), axis=0)

# Chuẩn hóa
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_filled)

print(f"✅ Dữ liệu phân cụm: {X_scaled.shape[0]} sinh viên × {X_scaled.shape[1]} môn học")


In [ ]:
# ── Elbow Method chọn K tối ưu ──
inertias = []
K_range = range(2, 9)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(K_range, inertias, 'bo-', linewidth=2, markersize=8)
ax.axvline(3, color='red', linestyle='--', label='K=3 (theo yêu cầu)')
ax.set_title('Elbow Method – Chọn số cụm K tối ưu', fontsize=13, fontweight='bold')
ax.set_xlabel('Số cụm K')
ax.set_ylabel('Inertia (WCSS)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('elbow_method.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# -- Phan cum K=3 --
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
labels = kmeans.fit_predict(X_scaled)
df_sv['Cum'] = labels

# Dat ten nhom theo GPA trung binh
cum_gpa = df_sv.groupby('Cum')['GPA'].mean().sort_values(ascending=False)
nhom_labels = ['Nhom A - Hoc luc Tot', 'Nhom B - Hoc luc Trung binh', 'Nhom C - Can cai thien']
cum_map = {cum_gpa.index[j]: nhom_labels[j] for j in range(3)}
df_sv['Nhom'] = df_sv['Cum'].map(cum_map)

# Thong ke tung nhom
summary = df_sv.groupby('Nhom').agg(
    So_SV=('MSSV','count'),
    GPA_TB=('GPA','mean'),
    GPA_Max=('GPA','max'),
    GPA_Min=('GPA','min')
).round(2)
print("Thong ke 3 nhom phan cum:")
print(summary)


In [ ]:
# -- Truc quan hoa voi PCA 2D --
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

colors_cluster = {
    'Nhom A - Hoc luc Tot': '#2ecc71',
    'Nhom B - Hoc luc Trung binh': '#f39c12',
    'Nhom C - Can cai thien': '#e74c3c'
}

fig, ax = plt.subplots(figsize=(12, 7))

for nhom, color in colors_cluster.items():
    mask = df_sv['Nhom'] == nhom
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
               label=f"{nhom} (n={mask.sum()})",
               color=color, s=100, alpha=0.8, edgecolors='white', linewidths=0.8)
    for idx in df_sv[mask].index:
        name_short = str(df_sv.loc[idx, 'Ho_Ten']).split()[-1]
        ax.annotate(name_short, (X_pca[idx,0], X_pca[idx,1]),
                    fontsize=6.5, alpha=0.7, xytext=(3,3), textcoords='offset points')

centers_pca = pca.transform(kmeans.cluster_centers_)
ax.scatter(centers_pca[:,0], centers_pca[:,1],
           c='black', s=200, marker='X', zorder=5, label='Tam cum')

var_total = pca.explained_variance_ratio_.sum()*100
var_pc1   = pca.explained_variance_ratio_[0]*100
var_pc2   = pca.explained_variance_ratio_[1]*100
ax.set_title(f'Phan cum K-Means K=3 - PCA 2D ({var_total:.1f}% phuong sai)', fontsize=13, fontweight='bold')
ax.set_xlabel(f'PC1 ({var_pc1:.1f}%)')
ax.set_ylabel(f'PC2 ({var_pc2:.1f}%)')
ax.legend(loc='upper right')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('phan_cum_kmeans.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Variance explained: {var_total:.1f}%")


In [ ]:
# ── Danh sách sinh viên từng nhóm ──
for nhom in ['Nhóm A – Học lực Tốt', 'Nhóm B – Học lực Trung bình', 'Nhóm C – Cần cải thiện']:
    sv_nhom = df_sv[df_sv['Nhom']==nhom][['MSSV','Ho_Ten','GPA']].sort_values('GPA',ascending=False)
    sv_nhom = sv_nhom.reset_index(drop=True)
    sv_nhom.index += 1
    print(f"\n{'='*55}")
    print(f" {nhom} ({len(sv_nhom)} SV)")
    print('='*55)
    print(sv_nhom.to_string())


## 6. 🧠 Xây dựng mô hình dự đoán GPA

> Dùng hồi quy tuyến tính để dự đoán GPA cuối khoá từ điểm các môn năm đầu.


In [ ]:
# ── Môn năm 1 & 2 (feature) → GPA tổng (target) ──
mon_nam1 = ['ENG112','BAS123','BAS0108','BAS111','BAS215','TEE0103','BAS109']
mon_nam2 = ['BAS0205','BAS217','BAS305','TEE307','TEE319','TEE415',
            'TEE416','TEE423','TEE0211','TEE306']
feature_cols = [m for m in mon_nam1 + mon_nam2 if m in df_sv.columns]

df_model = df_sv[feature_cols + ['GPA']].dropna()
print(f"✅ Dữ liệu mô hình: {len(df_model)} mẫu × {len(feature_cols)} đặc trưng")

X = df_model[feature_cols].values
y = df_model['GPA'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
r2  = r2_score(y_test, y_pred)
print(f"\n📐 Kết quả mô hình Linear Regression:")
print(f"   MSE  = {mse:.4f}")
print(f"   RMSE = {np.sqrt(mse):.4f}")
print(f"   R²   = {r2:.4f}")


In [ ]:
# ── Biểu đồ Actual vs Predicted ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter actual vs predicted
axes[0].scatter(y_test, y_pred, color='steelblue', alpha=0.8, edgecolors='white', s=80)
axes[0].plot([y.min(), y.max()], [y.min(), y.max()], 'r--', lw=2, label='Dự đoán hoàn hảo')
axes[0].set_xlabel('GPA thực tế')
axes[0].set_ylabel('GPA dự đoán')
axes[0].set_title(f'Actual vs Predicted\n(R² = {r2:.3f})', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Feature importance (hệ số hồi quy)
coef_df = pd.Series(model.coef_, index=feature_cols).sort_values(key=abs, ascending=True)
mon_labels = [str(mon_dict.get(m, m))[:25] for m in coef_df.index]
colors_coef = ['#e74c3c' if v < 0 else '#2ecc71' for v in coef_df.values]
axes[1].barh(mon_labels, coef_df.values, color=colors_coef, edgecolor='white', alpha=0.85)
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_title('Hệ số hồi quy (Feature Importance)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Hệ số β')
axes[1].grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('model_du_doan.png', dpi=150, bbox_inches='tight')
plt.show()


## 7. 👤 Phân tích cá nhân – Lương Quang Hà (K225480106010)

In [ ]:
# ── Lấy dữ liệu của Lương Quang Hà ──
my_mssv = 'K225480106010'
my_row = df_sv[df_sv['MSSV'] == my_mssv].iloc[0]

my_scores = pd.Series({
    mon_dict.get(m, m): my_row[m]
    for m in mon_cols
    if not pd.isna(my_row[m])
})

print(f"👤 Sinh viên: {my_row['Ho_Ten']}")
print(f"🎓 MSSV: {my_row['MSSV']}")
print(f"📊 GPA trung bình: {my_row['GPA']:.2f}")
print(f"🏅 Xếp loại: {my_row['Xep_loai']}")
print(f"👥 Nhóm phân cụm: {my_row['Nhom']}")
print(f"📚 Số môn có điểm: {my_row['So_mon_co_diem']}")

# Xếp hạng trong lớp
rank = (df_sv['GPA'] > my_row['GPA']).sum() + 1
print(f"🏆 Xếp hạng trong lớp: {rank}/{len(df_sv)}")


In [ ]:
# ── Biểu đồ điểm cá nhân vs trung bình lớp ──
mon_compare = [m for m in mon_cols if not pd.isna(my_row[m])]
my_vals   = [my_row[m] for m in mon_compare]
avg_vals  = [df_sv[m].mean() for m in mon_compare]
labels_c  = [str(mon_dict.get(m,m))[:22] for m in mon_compare]

x = np.arange(len(mon_compare))
width = 0.38

fig, ax = plt.subplots(figsize=(16, 7))
bars1 = ax.bar(x - width/2, my_vals, width, label='Lương Quang Hà', color='#3498db', alpha=0.87, edgecolor='white')
bars2 = ax.bar(x + width/2, avg_vals, width, label='TB lớp', color='#95a5a6', alpha=0.75, edgecolor='white')

ax.set_xticks(x)
ax.set_xticklabels(labels_c, rotation=55, ha='right', fontsize=8)
ax.set_ylabel('Điểm (thang 4)')
ax.set_title('So sánh điểm Lương Quang Hà vs Trung bình lớp 58KTPM', fontsize=13, fontweight='bold')
ax.legend()
ax.axhline(2.0, color='red', linestyle=':', alpha=0.5, label='Ngưỡng 2.0')
ax.set_ylim(0, 4.5)
ax.grid(axis='y', alpha=0.3)

# Đánh dấu môn vượt trội / yếu
for i, (mv, av) in enumerate(zip(my_vals, avg_vals)):
    diff = mv - av
    color = '#27ae60' if diff > 0.3 else '#e74c3c' if diff < -0.3 else 'gray'
    ax.annotate(f'{diff:+.1f}', (x[i], max(mv, av)+0.1), ha='center', fontsize=7, color=color, fontweight='bold')

plt.tight_layout()
plt.savefig('phan_tich_ca_nhan.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── Radar Chart – 6 nhóm môn ──
nhom_mon = {
    'Lập trình\nchuyên ngành': ['TEE0432','TEE319','TEE307','TEE0479','TEE0103'],
    'Hệ thống &\nMạng': ['TEE416','TEE433','TEE435','TEE306','TEE318'],
    'AI & Dữ liệu': ['TEE0591','TEE321','TEE423','TEE560'],
    'Kỹ thuật &\nPhần cứng': ['TEE403','TEE408','TEE415','TEE0217','TEE314'],
    'Quản lý &\nDự án': ['TEE0480','TEE0497','TEE567','TEE563','WSH442'],
    'Đại cương &\nAnh văn': ['BAS0108','BAS111','BAS0205','BAS305','BAS123','ENG112','ENG113','ENG217']
}

def nhom_avg(mssv_row, mons):
    vals = [mssv_row[m] for m in mons if m in df_sv.columns and not pd.isna(mssv_row.get(m, np.nan))]
    return np.mean(vals) if vals else 0

my_radar   = [nhom_avg(my_row,     v) for v in nhom_mon.values()]
avg_radar  = [df_sv.apply(lambda r: nhom_avg(r, v), axis=1).mean() for v in nhom_mon.values()]
labels_r   = list(nhom_mon.keys())

N = len(labels_r)
angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
angles += angles[:1]

my_radar  = my_radar  + my_radar[:1]
avg_radar = avg_radar + avg_radar[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
ax.plot(angles, my_radar,  'b-o', lw=2.5, label='Lương Quang Hà', markersize=6)
ax.fill(angles, my_radar,  'b', alpha=0.15)
ax.plot(angles, avg_radar, 'g--s', lw=2, label='TB lớp', markersize=5)
ax.fill(angles, avg_radar, 'g', alpha=0.1)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(labels_r, fontsize=10)
ax.set_ylim(0, 4)
ax.set_yticks([1, 2, 3, 4])
ax.set_title('Radar Chart – Năng lực theo nhóm môn', fontsize=13, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig('radar_chart.png', dpi=150, bbox_inches='tight')
plt.show()


## 8. 📝 Tổng kết & Nhận xét

In [ ]:
print("=" * 60)
print("  📊 BÁO CÁO TỔNG KẾT – LỚP 58KTPM")
print("=" * 60)

total = len(df_sv)
gpa_tb = df_sv['GPA'].mean()

print(f"\n🏫 Tổng số sinh viên phân tích: {total}")
print(f"📈 GPA trung bình toàn lớp    : {gpa_tb:.2f}/4.0")
print(f"🏆 GPA cao nhất               : {df_sv['GPA'].max():.2f} ({df_sv.loc[df_sv['GPA'].idxmax(),'Ho_Ten']})")
print(f"📉 GPA thấp nhất              : {df_sv['GPA'].min():.2f}")

print(f"\n📋 Phân bố xếp loại:")
for xl, count in df_sv['Xep_loai'].value_counts().items():
    pct = count/total*100
    print(f"   {xl:<20}: {count:>3} SV ({pct:.1f}%)")

print(f"\n👥 Kết quả phân cụm K-Means (K=3):")
for nhom, count in df_sv['Nhom'].value_counts().items():
    gpa_n = df_sv[df_sv['Nhom']==nhom]['GPA'].mean()
    print(f"   {nhom}: {count} SV | GPA TB = {gpa_n:.2f}")

print(f"\n👤 Thông tin bản thân:")
print(f"   Họ tên : Lương Quang Hà")
print(f"   MSSV   : K225480106010")
print(f"   GPA    : {my_row['GPA']:.2f} | Xếp loại: {my_row['Xep_loai']}")
print(f"   Nhóm   : {my_row['Nhom']}")
print(f"   Xếp hạng: {rank}/{total} trong lớp")

print(f"\n🤖 Mô hình dự đoán Linear Regression:")
print(f"   R² Score = {r2:.4f} | RMSE = {np.sqrt(mse):.4f}")
print("=" * 60)


---
## 📌 Kết luận

1. **Phân tích tổng thể:** Lớp 58KTPM có GPA trung bình phản ánh đặc thù chương trình CNPM với nhiều môn kỹ thuật khó.
2. **Phân cụm K-Means K=3:** Chia lớp thành 3 nhóm rõ rệt giúp giảng viên có chiến lược hỗ trợ phù hợp.
3. **Mô hình dự đoán:** Linear Regression từ điểm năm 1-2 có thể ước tính GPA tổng kết với độ chính xác nhất định.
4. **Cá nhân Lương Quang Hà:** Có điểm mạnh ở các môn lập trình chuyên ngành (Python, AI/ML), còn cần cải thiện ở các môn phần cứng và toán cơ sở.

> *Notebook được tạo cho bài tập Python Data Science – Lớp 58KTPM*
